In [1]:
!pip install xgboost --quiet

# *Model Training* 

In [4]:
# Importing Libraries
import pandas as pd
import numpy as np
import os
import joblib
import warnings

warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, accuracy_score

from xgboost import XGBClassifier

print("Libraries Loaded Successfully")

Libraries Loaded Successfully


In [5]:
# GPU Check 
import tensorflow as tf

print("TensorFlow Version:", tf.__version__)

gpus = tf.config.list_physical_devices('GPU')

if gpus:
    print("GPU IS ACTIVE")
    print(gpus)
else:
    print("GPU NOT DETECTED")

2026-05-13 14:41:36.556131: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778683296.805950      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778683296.876811      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778683297.437966      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778683297.438014      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778683297.438017      57 computation_placer.cc:177] computation placer alr

TensorFlow Version: 2.19.0
GPU IS ACTIVE
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [6]:
# Dataset Path

data_path = "/kaggle/input/datasets/galaxyh/kdd-cup-1999-data/kddcup.data_10_percent.gz"

# Column Names

columns = [
    "duration","protocol_type","service","flag","src_bytes",
    "dst_bytes","land","wrong_fragment","urgent","hot",
    "num_failed_logins","logged_in","num_compromised","root_shell",
    "su_attempted","num_root","num_file_creations","num_shells",
    "num_access_files","num_outbound_cmds","is_host_login",
    "is_guest_login","count","srv_count","serror_rate",
    "srv_serror_rate","rerror_rate","srv_rerror_rate",
    "same_srv_rate","diff_srv_rate","srv_diff_host_rate",
    "dst_host_count","dst_host_srv_count",
    "dst_host_same_srv_rate","dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate","dst_host_srv_diff_host_rate",
    "dst_host_serror_rate","dst_host_srv_serror_rate",
    "dst_host_rerror_rate","dst_host_srv_rerror_rate",
    "label"
]

# Load Data

df = pd.read_csv(data_path, names=columns)

print("Dataset Loaded")
print(df.shape)

df.head()

Dataset Loaded
(494021, 42)


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label
0,0,tcp,http,SF,181,5450,0,0,0,0,...,9,1.0,0.0,0.11,0.0,0.0,0.0,0.0,0.0,normal.
1,0,tcp,http,SF,239,486,0,0,0,0,...,19,1.0,0.0,0.05,0.0,0.0,0.0,0.0,0.0,normal.
2,0,tcp,http,SF,235,1337,0,0,0,0,...,29,1.0,0.0,0.03,0.0,0.0,0.0,0.0,0.0,normal.
3,0,tcp,http,SF,219,1337,0,0,0,0,...,39,1.0,0.0,0.03,0.0,0.0,0.0,0.0,0.0,normal.
4,0,tcp,http,SF,217,2032,0,0,0,0,...,49,1.0,0.0,0.02,0.0,0.0,0.0,0.0,0.0,normal.


In [7]:
# labels

df['label'] = df['label'].apply(
    lambda x: 'normal' if x == 'normal.' else 'attack'
)

df['label'].value_counts()

label
attack    396743
normal     97278
Name: count, dtype: int64

In [8]:
# Label Encoder
categorical_columns = ['protocol_type', 'service', 'flag']

encoders = {}

for col in categorical_columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

print("Categorical Encoding Done")

Categorical Encoding Done


In [9]:
 # target encoder
target_encoder = LabelEncoder()

df['label'] = target_encoder.fit_transform(df['label'])

print(target_encoder.classes_)

['attack' 'normal']


In [10]:
X = df.drop("label", axis=1)
y = df["label"]

print(X.shape)
print(y.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train Shape:", X_train.shape)
print("Test Shape:", X_test.shape)

(494021, 41)
(494021,)
Train Shape: (395216, 41)
Test Shape: (98805, 41)


In [11]:
# Feature Scaling 
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Scaling Complete")

Scaling Complete


In [12]:
# Training 

model = XGBClassifier(
    n_estimators=200,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='binary:logistic',
    tree_method='hist',   # GPU Training
    predictor='gpu_predictor',
    eval_metric='logloss',
    random_state=42
)

model.fit(X_train, y_train)

print("MODEL TRAINING COMPLETED")

MODEL TRAINING COMPLETED


In [13]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.9998481858205557

Classification Report:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     79349
           1       1.00      1.00      1.00     19456

    accuracy                           1.00     98805
   macro avg       1.00      1.00      1.00     98805
weighted avg       1.00      1.00      1.00     98805



In [14]:
os.makedirs("/kaggle/working/saved_model", exist_ok=True)

# Save Model
joblib.dump(model, "/kaggle/working/saved_model/model.pkl")

# Save Scaler
joblib.dump(scaler, "/kaggle/working/saved_model/scaler.pkl")

# Save Encoders
joblib.dump(encoders, "/kaggle/working/saved_model/encoders.pkl")

# Save Target Encoder
joblib.dump(target_encoder, "/kaggle/working/saved_model/target_encoder.pkl")

# Save Feature Names
joblib.dump(X.columns.tolist(), "/kaggle/working/saved_model/features.pkl")

print("ALL FILES SAVED")

ALL FILES SAVED


In [15]:
import shutil

shutil.make_archive(
    "/kaggle/working/network_ids_model",
    'zip',
    "/kaggle/working/saved_model"
)

print("ZIP FILE CREATED")

ZIP FILE CREATED


In [16]:
loaded_model = joblib.load(
    "/kaggle/working/saved_model/model.pkl"
)

sample_prediction = loaded_model.predict(X_test[:5])

print(sample_prediction)

[0 1 0 0 0]


In [17]:
import os

print(os.listdir("/kaggle/working"))

['network_ids_model.zip', 'saved_model', '.virtual_documents']
